# Paso 5 — Validar la integridad del Data Warehouse
**Archivo original:** `etl/ConsAUX.py`

---

## ¿Qué hace este script?

Después de cargar todos los datos, verificamos que el DW esté **íntegro y consistente**.
Este paso es fundamental en cualquier proyecto de datos real.

**8 validaciones que se realizan:**
1. Listar todas las tablas del esquema `dw`
2. Contar filas por tabla
3. Detectar duplicados en la fact table
4. Buscar NULLs en la clave natural de la fact table
5. Detectar claves foráneas huérfanas (FK sin match en la dimensión)
6. Detectar duplicados en cada dimensión
7. Validación especial de `dim_producto` (clave compuesta)
8. Validación básica del staging


In [ ]:
import duckdb

DB_PATH = r"C:\Información\proyectos\aduana_bi\db\aduana.duckdb"
con = duckdb.connect(DB_PATH)

print("=============================")
print(" VALIDACIÓN DEL MODELO DW    ")
print("=============================")

---

## Validación 1: Tablas existentes

`information_schema.tables` es una vista del sistema que lista todas las tablas.
Sirve para confirmar que el script de creación funcionó correctamente.


In [ ]:
tables = con.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'dw'
    ORDER BY table_name
""").fetchall()

print(f"\n1) Tablas en el esquema 'dw': {len(tables)}\n")
for t in tables:
    print(f"   - {t[0]}")

---

## Validación 2: Conteo de filas por tabla

Permite detectar tablas vacías o con volumen inesperado.
Un DW bien cargado debería tener:
- Staging: igual cantidad que el Excel fuente
- Dimensiones: mucho menos filas que el staging
- Fact table: igual cantidad que el staging (menos filas con clave nula)


In [ ]:
print("\n2) Conteo de filas por tabla:\n")
print(f"{'Tabla':<35} {'Filas':>8}")
print("-" * 45)
for t in tables:
    table_name = t[0]
    count = con.execute(f"SELECT COUNT(*) FROM dw.{table_name}").fetchone()[0]
    print(f"   {table_name:<32} {count:>8}")

---

## Validación 3: Duplicados en la fact table

La clave natural `(despacho_cifrado, item)` debe ser única.
Si hay duplicados, significa que el mismo ítem se insertó dos veces.

**Técnica SQL:** `GROUP BY + HAVING COUNT(*) > 1`
- `GROUP BY` agrupa las filas con la misma clave
- `HAVING COUNT(*) > 1` filtra solo los grupos con más de una fila (duplicados)


In [ ]:
dup_fact = con.execute("""
    SELECT COUNT(*)
    FROM (
        SELECT despacho_cifrado, item, COUNT(*) c
        FROM dw.fact_aduana_item
        GROUP BY despacho_cifrado, item
        HAVING COUNT(*) > 1   -- solo grupos con duplicados
    ) x
""").fetchone()[0]

estado = "OK" if dup_fact == 0 else f"ALERTA: {dup_fact} duplicados"
print(f"\n3) Duplicados en fact_aduana_item: {estado}")

---

## Validación 4: NULLs en clave natural de la fact table

Si `despacho_cifrado` o `item` son NULL, esa fila no tiene identidad
y no puede relacionarse con otros sistemas.


In [ ]:
nulls = con.execute("""
    SELECT
        SUM(CASE WHEN despacho_cifrado IS NULL THEN 1 ELSE 0 END) AS null_despacho,
        SUM(CASE WHEN item IS NULL THEN 1 ELSE 0 END) AS null_item
    FROM dw.fact_aduana_item
""").fetchone()

print(f"\n4) NULLs en clave natural:")
print(f"   despacho_cifrado NULL: {nulls[0]}")
print(f"   item NULL:             {nulls[1]}")

---

## Validación 5: Claves foráneas huérfanas

Una **clave huérfana** es una FK en la fact table que no tiene match en la dimensión.
Por ejemplo: `id_operacion = 5` pero `dim_operacion` solo tiene IDs 1 y 2.

**Técnica SQL:** `LEFT JOIN` + `WHERE d.pk IS NULL AND f.fk IS NOT NULL`
- Si el LEFT JOIN no encuentra match, todas las columnas de la dimensión quedan NULL
- Filtramos esas filas para detectar las huérfanas


In [ ]:
fk_dim_pk = {
    "id_operacion"           : ("dim_operacion",         "id_operacion"),
    "id_destinacion"         : ("dim_destinacion",       "id_destinacion"),
    "id_regimen"             : ("dim_regimen",           "id_regimen"),
    "id_aduana"              : ("dim_aduana",            "id_aduana"),
    "id_pais_origen"         : ("dim_pais",              "id_pais"),
    "id_pais_destino"        : ("dim_pais",              "id_pais"),
    "id_producto"            : ("dim_producto",          "id_producto"),
    "id_medio_transporte"    : ("dim_medio_transporte",  "id_medio_transporte"),
    "id_canal"               : ("dim_canal",             "id_canal"),
    "id_unidad_medida"       : ("dim_unidad_medida",     "id_unidad_medida"),
    "id_acuerdo"             : ("dim_acuerdo",           "id_acuerdo"),
    "id_marca"               : ("dim_marca",             "id_marca"),
    "id_fecha_oficializacion": ("dim_fecha",             "id_fecha"),
    "id_fecha_cancelacion"   : ("dim_fecha",             "id_fecha"),
}

print("\n5) Claves foráneas huérfanas:\n")
print(f"{'FK':<28} {'Dimensión':<25} {'Huérfanas':>10}")
print("-" * 65)

for fk, (dim, pk) in fk_dim_pk.items():
    count = con.execute(f"""
        SELECT COUNT(*)
        FROM dw.fact_aduana_item f
        LEFT JOIN dw.{dim} d ON f.{fk} = d.{pk}
        WHERE f.{fk} IS NOT NULL   -- solo si la FK tiene valor
          AND d.{pk} IS NULL       -- pero no encontró match en la dimensión
    """).fetchone()[0]
    estado = str(count) if count == 0 else f"ALERTA: {count}"
    print(f"   {fk:<25} {dim:<25} {estado:>10}")

---

## Validación 6: Duplicados en dimensiones

Cada dimensión debe tener una sola fila por valor único (clave natural).
Si hay duplicados en una dimensión, los JOINs de la fact table podrían
multiplicar filas inesperadamente (fenómeno llamado "fan-out").


In [ ]:
dim_keys = {
    "dim_operacion"       : "operacion",
    "dim_destinacion"     : "cod_destinacion",
    "dim_regimen"         : "regimen",
    "dim_aduana"          : "aduana",
    "dim_pais"            : "codigo_pais",
    "dim_medio_transporte": "medio_transporte",
    "dim_canal"           : "canal",
    "dim_unidad_medida"   : "unidad_medida",
    "dim_acuerdo"         : "acuerdo",
    "dim_marca"           : "marca",
    "dim_fecha"           : "fecha",
}

print("\n6) Duplicados en dimensiones:\n")
for dim, key in dim_keys.items():
    dup = con.execute(f"""
        SELECT COUNT(*)
        FROM (
            SELECT {key}, COUNT(*) c
            FROM dw.{dim}
            GROUP BY {key}
            HAVING COUNT(*) > 1
        ) x
    """).fetchone()[0]
    estado = "OK" if dup == 0 else f"ALERTA: {dup}"
    print(f"   {dim:<30} {estado}")

---

## Validación 7: dim_producto — clave compuesta

`dim_producto` no tiene un solo campo único, sino 6 campos combinados.
Verificamos que ninguna combinación de los 6 aparezca más de una vez.


In [ ]:
dup_prod = con.execute("""
    SELECT COUNT(*)
    FROM (
        SELECT posicion_ncm, rubro, desc_capitulo, desc_posicion, desc_partida, mercaderia,
               COUNT(*) c
        FROM dw.dim_producto
        GROUP BY posicion_ncm, rubro, desc_capitulo, desc_posicion, desc_partida, mercaderia
        HAVING COUNT(*) > 1
    ) x
""").fetchone()[0]

estado = "OK" if dup_prod == 0 else f"ALERTA: {dup_prod} combinaciones duplicadas"
print(f"\n7) dim_producto (clave compuesta): {estado}")

---

## Validación 8: Staging básico


In [ ]:
stg = con.execute("""
    SELECT
        COUNT(*) AS total_filas,
        SUM(CASE WHEN despacho_cifrado IS NULL THEN 1 ELSE 0 END) AS null_despacho,
        SUM(CASE WHEN item IS NULL THEN 1 ELSE 0 END)             AS null_item,
        SUM(CASE WHEN oficializacion IS NULL THEN 1 ELSE 0 END)   AS null_fecha
    FROM dw.stg_aduana
""").fetchone()

print(f"\n8) Staging:")
print(f"   Total filas:             {stg[0]}")
print(f"   despacho_cifrado NULL:   {stg[1]}")
print(f"   item NULL:               {stg[2]}")
print(f"   oficializacion NULL:     {stg[3]}")

con.close()
print("\nValidación completada.")

---

## ¿Qué significan los resultados ideales?

| Validación | Resultado esperado |
|------------|--------------------|
| Tablas existentes | 15 tablas |
| stg_aduana | = filas del Excel |
| fact_aduana_item | = filas del staging (sin NULLs en clave) |
| Duplicados en fact | 0 |
| NULLs en clave | 0 |
| Claves huérfanas | 0 en todas |
| Duplicados en dims | 0 en todas |
| Duplicados en producto | 0 |

---

**El DW está listo para conectarse a Power BI.**

Conexión: Power BI → Obtener datos → ODBC → DSN apuntando a `aduana.duckdb`
